<a href="https://colab.research.google.com/github/Solo7602/web/blob/2lab/web_lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install requests pandas numpy matplotlib folium tqdm python-dateutil

In [4]:
import os
import json
import re
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.pipeline import make_pipeline
import joblib

import matplotlib.pyplot as plt
import seaborn as sns

# ---------- Config ----------
# Path to CSV file (adjust if needed)
DEFAULT_CSV = "sample_data/hh_vacancies.csv"
OUT_DIR = "content/sample_data/hh_ml_outputs"
RANDOM_STATE = 42
TEST_SIZE = 0.25

# If no label in data, these keyword maps used to form heuristic labels (edit as you want)
HEURISTIC_LABELS = {
    "system_analyst": ["системн", "system analyst", "system-analyst", "systemanalyst", "systems analyst"],
    "analyst": ["аналитик", "аналитик данных", "data analyst", "analyst", "bi", "business intelligence", "data"],
    "developer": ["разработчик", "developer", "java", "python", "c#", "c++", "dotnet", "разработ"],
    # add more categories if needed
}
# ---------------------------

os.makedirs(OUT_DIR, exist_ok=True)
warnings.filterwarnings("ignore")

def load_csv(path):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"CSV not found at {path}. Поместите файл и повторите.")
    df = pd.read_csv(p)
    print(f"Loaded {p} with shape {df.shape}")
    return df

def build_text_column(df):
    # Try to find a sensible text column: prefer 'text', else combine name+description+skills+raw
    possible_text_cols = [c for c in df.columns if c.lower() in ("text","description","vacancy","vacancy_text","job_description")]
    if possible_text_cols:
        print("Using existing text column:", possible_text_cols[0])
        df["text"] = df[possible_text_cols[0]].fillna("").astype(str)
        return df

    # Combine available fields
    parts = []
    for col in ("name","title","position","description","key_skills","raw","responsibility","duties"):
        if col in df.columns:
            parts.append(col)
    if not parts:
        # fallback: just stringify each row
        df["text"] = df.astype(str).agg(" ".join, axis=1)
        print("No common text columns found — built 'text' by stringifying the row.")
        return df

    def combine_row(r):
        texts = []
        for c in parts:
            v = r.get(c)
            if pd.isna(v):
                continue
            # if key_skills might be a list-like string, keep it
            texts.append(str(v))
        return " ".join(texts).strip()

    df["text"] = df.apply(combine_row, axis=1)
    print("Built 'text' from columns:", parts)
    return df

# Basic cleaning and optional russian lemmatization
token_re = re.compile(r"\b[\w\+\#\-\']+\b", flags=re.UNICODE)

def preprocess_text(text, lowercase=True, lemmatize=False):
    if not isinstance(text, str):
        text = str(text)
    if lowercase:
        text = text.lower()
    # remove HTML tags if present
    text = re.sub(r"<[^>]+>", " ", text)
    # keep tokens
    tokens = token_re.findall(text)
    if lemmatize and MORPH:
        lemmas = []
        for t in tokens:
            # handle english words too - pymorphy will leave them as-is
            try:
                p = MORPH.parse(t)[0]
                lemmas.append(p.normal_form)
            except Exception:
                lemmas.append(t)
        return " ".join(lemmas)
    else:
        return " ".join(tokens)

def build_labels(df):
    # If label column exists, use it
    label_cols = [c for c in df.columns if c.lower() in ("label","target","category","profession","job_type","class")]
    if label_cols:
        labcol = label_cols[0]
        print("Using existing label column:", labcol)
        df["label"] = df[labcol].astype(str).fillna("unknown")
        return df, True

    # Otherwise build heuristic labels using keywords
    print("No label column found — building heuristic labels based on keywords.")
    def assign_label(text):
        t = str(text).lower()
        for lab, keywords in HEURISTIC_LABELS.items():
            for kw in keywords:
                if kw in t:
                    return lab
        return "other"

    df["label"] = df["text"].fillna("").apply(assign_label)
    # report distribution
    print("Heuristic label distribution:", Counter(df["label"]))
    return df, False

def safe_train_eval(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE):
    # ensure enough samples per class for stratify. if not - skip stratify
    stratify = y if len(set(y)) > 1 and min(Counter(y).values()) >= 2 else None
    if stratify is None:
        print("Not enough samples per class for stratified split -> doing random split without stratify.")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=stratify)
    return X_train, X_test, y_train, y_test

def train_classifiers(X_train_tfidf, X_test_tfidf, y_train, y_test, out_dir):
    results = {}
    # Logistic Regression
    lr = LogisticRegression(max_iter=2000, solver="liblinear", random_state=RANDOM_STATE)
    lr.fit(X_train_tfidf, y_train)
    y_pred = lr.predict(X_test_tfidf)
    results["logreg"] = {
        "model": lr,
        "y_pred": y_pred,
    }
    # Save
    joblib.dump(lr, os.path.join(out_dir, "logreg_model.joblib"))
    print("Saved LogisticRegression model.")

    # Optionally, you can add RandomForest or XGBoost here.

    return results

def eval_and_save(y_test, y_pred, out_dir, label_names=None):
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    report = classification_report(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    summary = {
        "accuracy": float(acc),
        "f1_weighted": float(f1),
        "precision_weighted": float(prec),
        "recall_weighted": float(rec),
        "classification_report": report,
        "confusion_matrix": cm.tolist()
    }
    with open(os.path.join(out_dir, "classification_summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    # Plot confusion matrix
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=label_names, yticklabels=label_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion matrix")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "confusion_matrix.png"))
    plt.close()

    print("Classification results saved to", out_dir)
    print("Accuracy:", acc)
    print(report)

    return summary

def lda_topic_modeling(texts, n_topics=8, out_dir=OUT_DIR):
    # Count vectorizer
    vectorizer = CountVectorizer(max_features=5000, token_pattern=r"(?u)\b\w\w+\b")
    X_counts = vectorizer.fit_transform(texts)
    lda = LatentDirichletAllocation(n_components=n_topics, random_state=RANDOM_STATE, learning_method="batch", max_iter=30)
    lda.fit(X_counts)

    feature_names = vectorizer.get_feature_names_out()
    topics = []
    for topic_idx, topic in enumerate(lda.components_):
        top_features_ind = topic.argsort()[:-11:-1]
        top_features = [feature_names[i] for i in top_features_ind]
        topics.append(top_features)
        print(f"Topic {topic_idx}: {', '.join(top_features)}")

    # Assign topic to each document
    doc_topic = lda.transform(X_counts)
    assigned = np.argmax(doc_topic, axis=1)

    # Save topics
    with open(os.path.join(out_dir, "lda_topics.json"), "w", encoding="utf-8") as f:
        json.dump({"topics": topics}, f, ensure_ascii=False, indent=2)

    return assigned, topics

def bertopic_modeling(texts, out_dir=OUT_DIR):
    if not BERTOPIC_AVAILABLE:
        print("BERTopic not available (not installed). Skipping.")
        return None, None
    print("Running BERTopic (this requires sentence-transformers and may be slow)...")
    # If lots of text, you may need to set embedding model to something smaller
    topic_model = BERTopic(language="multilingual", calculate_probabilities=True, verbose=False)
    topics, probs = topic_model.fit_transform(texts)
    topic_model.get_topic_info().to_csv(os.path.join(out_dir, "bertopic_info.csv"), index=False)
    topic_model.save(os.path.join(out_dir, "bertopic_model"))
    return topics, topic_model

def auto_conclusions(classification_summary, topics_list, label_names):
    # Build simple textual conclusions based on metrics and topics
    lines = []
    acc = classification_summary.get("accuracy")
    f1 = classification_summary.get("f1_weighted")
    lines.append(f"Краткие выводы по классификатору: accuracy = {acc:.3f}, weighted F1 = {f1:.3f}.")
    if acc >= 0.85:
        lines.append("Модель демонстрирует хорошую точность. Можно рассмотреть использование в продакшен-процессе при добавлении валидации.")
    elif acc >= 0.6:
        lines.append("Модель показывает умеренную точность — рекомендуется дообучение на большей размеченной выборке и/или использовать более мощные модели (BERT-подобные).")
    else:
        lines.append("Модель пока недостаточно точна — вероятная причина: малая/шумная выборка, некачественная разметка или слишком простая модель.")
    # topic notes
    if topics_list:
        lines.append(f"Выделено {len(topics_list)} тем. Топ слова по темам (первые 5 тем):")
        for i,t in enumerate(topics_list[:5]):
            lines.append(f"  Тема {i}: {', '.join(t[:8])}")
    else:
        lines.append("Тематическое моделирование не было выполнено.")
    # label distribution
    if label_names:
        lines.append(f"Найденные классы: {', '.join(map(str,label_names))}")
    return "\n".join(lines)

# --------- Main execution ----------
def main(csv_path=DEFAULT_CSV, lemmatize=False, n_lda_topics=8):
    df = load_csv(csv_path)
    df = build_text_column(df)
    df["text"] = df["text"].fillna("").astype(str)
    # Preprocess
    print("Preprocessing texts (lowercase tokenization; lemmatize=%s)" % lemmatize)
    df["text_proc"] = df["text"].apply(lambda t: preprocess_text(t, lowercase=True, lemmatize=lemmatize))

    # Labels
    df, had_label = build_labels(df)
    # ensure non-empty texts
    df = df[df["text_proc"].str.strip().astype(bool)].reset_index(drop=True)
    if df.empty:
        raise ValueError("Нет текстов для обработки после предобработки.")

    # Optionally map labels to integers
    label_names = sorted(list(df["label"].unique()))
    label_to_idx = {l:i for i,l in enumerate(label_names)}
    y = df["label"].map(label_to_idx).values

    # Vectorize (TF-IDF)
    tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2), token_pattern=r"(?u)\b\w\w+\b")
    X = tfidf.fit_transform(df["text_proc"])

    # Train/test split
    X_train, X_test, y_train, y_test = safe_train_eval(X, y, test_size=TEST_SIZE)
    print("Train size:", X_train.shape[0], "Test size:", X_test.shape[0])

    # Train model(s)
    models = train_classifiers(X_train, X_test, y_train, y_test, OUT_DIR)
    lr_pred = models["logreg"]["y_pred"]

    # Evaluate
    inv_label_names = [l for l in label_names]
    classification_summary = eval_and_save(y_test, lr_pred, OUT_DIR, label_names=inv_label_names)

    # Save TF-IDF vectorizer
    joblib.dump(tfidf, os.path.join(OUT_DIR, "tfidf_vectorizer.joblib"))

    # Topic modeling: LDA
    n_topics = min(n_lda_topics, max(2, int(len(df)/10)))  # heuristic: not more than len/10
    print(f"Running LDA with n_topics={n_topics}")
    lda_assigned, lda_topics = lda_topic_modeling(df["text_proc"].tolist(), n_topics=n_topics, out_dir=OUT_DIR)
    df["lda_topic"] = lda_assigned
    df.to_csv(os.path.join(OUT_DIR, "hh_ml_dataset_with_topics.csv"), index=False)


    # Compose automatic conclusions
    conclusions = auto_conclusions(classification_summary, lda_topics, label_names)
    with open(os.path.join(OUT_DIR, "conclusions.txt"), "w", encoding="utf-8") as f:
        f.write(conclusions)

    print("\n====== CONCLUSIONS ======\n")
    print(conclusions)
    print("\nAll outputs saved to", OUT_DIR)
    return {
        "df": df,
        "tfidf": tfidf,
        "model_logreg": models["logreg"]["model"],
        "classification_summary": classification_summary,
        "lda_topics": lda_topics
    }

if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser(description="HH ML pipeline")
    parser.add_argument("--csv", type=str, default=DEFAULT_CSV, help="Path to CSV file")
    parser.add_argument("--lemmatize", action="store_true", help="Use russian lemmatization via pymorphy2 if available")
    parser.add_argument("--lda_topics", type=int, default=8, help="Approx number of LDA topics")
    args, unknown = parser.parse_known_args()

    main(csv_path=args.csv, lemmatize=args.lemmatize, n_lda_topics=args.lda_topics)

Loaded sample_data/hh_vacancies.csv with shape (119, 14)
Built 'text' from columns: ['name', 'key_skills', 'raw']
Preprocessing texts (lowercase tokenization; lemmatize=False)
No label column found — building heuristic labels based on keywords.
Heuristic label distribution: Counter({'analyst': 88, 'system_analyst': 31})
Train size: 89 Test size: 30
Saved LogisticRegression model.
Classification results saved to content/sample_data/hh_ml_outputs
Accuracy: 0.7333333333333333
              precision    recall  f1-score   support

           0       0.73      1.00      0.85        22
           1       0.00      0.00      0.00         8

    accuracy                           0.73        30
   macro avg       0.37      0.50      0.42        30
weighted avg       0.54      0.73      0.62        30

Running LDA with n_topics=8
Topic 0: quot, gradient, not, 20px, 1с, мы, опыт, совкомбанк, and, u200b
Topic 1: name, none, id, false, hh, ru, https, true, анализ, на
Topic 2: name, none, false, id